# Solar Filament Segmentation - v3
Changes vs v2: **continues training from `work/model_v2.pt`** (no restart), larger training crops (more context), flip TTA on CPU,
and a much better post-processing search (adds a per-filament confidence filter, wider grid, cached IoUs so the search is fast).
Writes **`submission_v3.csv`** (your previous submission.csv is NOT overwritten).

%pip install -q segmentation-models-pytorch pycocotools opencv-python-headless pandas numpy tqdm

In [1]:
# 2) CONFIG
import os, json, glob, time, zipfile, itertools, csv, math
from collections import defaultdict
import numpy as np, cv2, pandas as pd, torch, torch.nn.functional as F
from tqdm.auto import tqdm
from pycocotools import mask as mu
import segmentation_models_pytorch as smp

ZIP_PATH = r"C:\Users\shrut\Desktop\final\filament-segmentation-2026.zip"
WORK_DIR = "./work"
RESUME_FROM = os.path.join(WORK_DIR, "model_v2.pt")   # set to None to train from scratch
TIME_BUDGET_MIN = None   # training minutes. None = auto (CPU 25, GPU 15)
SEED = 0                 # keep 0: same train/val split as v2, so the resumed model has not seen the val images

device = "cuda" if torch.cuda.is_available() else "cpu"
RES, EVAL_RES = 1024, 512
if device == "cuda":
    ENCODER, CROP, BS, BUDGET = "resnet34", 512, 8, (TIME_BUDGET_MIN or 15)*60
else:
    ENCODER, CROP, BS, BUDGET = "resnet18", 320, 6, (TIME_BUDGET_MIN or 25)*60
    torch.set_num_threads(os.cpu_count())
TTA = True               # horizontal-flip test-time augmentation (set False to halve inference time)
use_amp = device == "cuda"
print("device:", device, "| encoder", ENCODER, "| crop", CROP, "| train min:", BUDGET/60)
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)

c:\Users\shrut\Desktop\final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu | encoder resnet18 | crop 320 | train min: 25.0


In [2]:
# 3) unzip + locate files
data_dir = os.path.join(WORK_DIR, "data")
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(data_dir)
ann_path = glob.glob(os.path.join(data_dir, "**", "*Annotations*train*.json"), recursive=True)[0]
train_dir = [d for d in glob.glob(os.path.join(data_dir, "**", "train_images"), recursive=True) if os.path.isdir(d)][0]
test_dir  = [d for d in glob.glob(os.path.join(data_dir, "**", "test_images"),  recursive=True) if os.path.isdir(d)][0]
get_stem = lambda p: os.path.splitext(os.path.basename(p))[0][-16:]
train_files = {get_stem(p): p for p in glob.glob(os.path.join(train_dir, "*"))}
test_files  = {get_stem(p): p for p in sorted(glob.glob(os.path.join(test_dir, "*")))}
print(len(train_files), "train files,", len(test_files), "test files")

707 train files, 180 test files


In [3]:
# 4) dataset. Inputs = 2 channels: raw intensity + local contrast (image minus large-scale blur, removes limb darkening)
def make_inputs(gray, res=RES):
    x = cv2.resize(gray, (res, res), interpolation=cv2.INTER_AREA)
    blur = cv2.GaussianBlur(x.astype(np.float32), (0, 0), 30)
    d = np.clip((x.astype(np.float32) - blur) * 2 + 128, 0, 255).astype(np.uint8)
    return x, d

def to_tensor(x, d):   # uint8 arrays (..., H, W) -> float tensor (N, 2, H, W)
    x = torch.from_numpy(np.ascontiguousarray(x)).float(); d = torch.from_numpy(np.ascontiguousarray(d)).float()
    return torch.stack([(x/255 - 0.45)/0.25, (d - 128)/2/40], 1)

coco = json.load(open(ann_path))
anns_by_img = defaultdict(list)
for a in coco["annotations"]: anns_by_img[a["image_id"]].append(a)
groups = defaultdict(list)
for im in coco["images"]:
    s = get_stem(im["file_name"])
    if s in train_files: groups[s].append(im)
for s in groups: groups[s].sort(key=lambda d: str(d["id"]))
stems = sorted(groups); print("unique train images:", len(stems))
perm = rng.permutation(len(stems)); n_val = max(8, int(0.15*len(stems)))
val_stems = [stems[i] for i in perm[:n_val]]; tr_stems = [stems[i] for i in perm[n_val:]]

def poly_masks(img_info, res):   # one binary mask per filament, at resolution res
    sc = res / img_info["width"]; inst = []
    for a in anns_by_img[img_info["id"]]:
        m = np.zeros((res, res), np.uint8)
        for seg in a["segmentation"]:
            cv2.fillPoly(m, [np.round(np.array(seg, dtype=np.float64).reshape(-1, 2) * sc).astype(np.int32)], 1)
        inst.append(m)
    return inst

def union_soft(s, res):   # mean over annotators of their union mask, stored as uint8 0..255
    ms = []
    for im in groups[s]:
        p = poly_masks(im, res)
        ms.append(np.clip(np.sum(p, 0), 0, 1) if p else np.zeros((res, res), np.uint8))
    return np.round(np.mean(ms, 0) * 255).astype(np.uint8)

def read_gray(path): return cv2.imread(path, cv2.IMREAD_GRAYSCALE)

Xtr, Dtr, Ytr = [], [], []
for s in tqdm(tr_stems):
    x, d = make_inputs(read_gray(train_files[s])); Xtr.append(x); Dtr.append(d); Ytr.append(union_soft(s, RES))
Xtr, Dtr, Ytr = np.stack(Xtr), np.stack(Dtr), np.stack(Ytr)
fg_idx = [np.flatnonzero(y.ravel() > 127) for y in Ytr]   # foreground pixels, used to centre crops on filaments

Xva, Dva, GTva = [], [], []
for s in tqdm(val_stems):
    x, d = make_inputs(read_gray(train_files[s])); Xva.append(x); Dva.append(d)
    GTva.append([[m.astype(bool) for m in poly_masks(im, EVAL_RES)] for im in groups[s]])   # one instance list per annotator
Xva, Dva = np.stack(Xva), np.stack(Dva)
print(Xtr.shape, Xva.shape)

unique train images: 707


100%|██████████| 106/106 [00:05<00:00, 20.40it/s]

(601, 1024, 1024) (106, 1024, 1024)


In [4]:
# 5) model (warm start from v2 if available) + training on random crops
resume = RESUME_FROM is not None and os.path.exists(RESUME_FROM)
if resume:
    model = smp.Unet(ENCODER, encoder_weights=None, in_channels=2, classes=1)
    model.load_state_dict(torch.load(RESUME_FROM, map_location="cpu")); BASE_LR = 2e-4
    print("resumed from", RESUME_FROM)
else:
    try:
        model = smp.Unet(ENCODER, encoder_weights="imagenet", in_channels=2, classes=1); print("loaded ImageNet encoder")
    except Exception as e:
        print("pretrained weights unavailable -> from scratch"); model = smp.Unet(ENCODER, encoder_weights=None, in_channels=2, classes=1)
    BASE_LR = 5e-4
model.to(device)

def get_batch():
    xs, ds, ys = [], [], []
    for _ in range(BS):
        i = int(rng.integers(len(Xtr)))
        if rng.random() < 0.8 and len(fg_idx[i]):
            p = int(fg_idx[i][rng.integers(len(fg_idx[i]))]); cy, cx = divmod(p, RES)
            y0 = int(np.clip(cy - CROP//2 + rng.integers(-CROP//4, CROP//4 + 1), 0, RES - CROP))
            x0 = int(np.clip(cx - CROP//2 + rng.integers(-CROP//4, CROP//4 + 1), 0, RES - CROP))
        else:
            y0, x0 = int(rng.integers(0, RES - CROP + 1)), int(rng.integers(0, RES - CROP + 1))
        sl = (slice(y0, y0 + CROP), slice(x0, x0 + CROP))
        a, b, c = Xtr[i][sl], Dtr[i][sl], Ytr[i][sl]
        if rng.random() < .5: a, b, c = a[:, ::-1], b[:, ::-1], c[:, ::-1]
        if rng.random() < .5: a, b, c = a[::-1], b[::-1], c[::-1]
        k = int(rng.integers(4)); a, b, c = np.rot90(a, k), np.rot90(b, k), np.rot90(c, k)
        xs.append(a); ds.append(b); ys.append(c)
    t = to_tensor(np.stack(xs), np.stack(ds))
    t[:, 0] = t[:, 0] * float(rng.uniform(0.9, 1.1)) + float(rng.uniform(-0.1, 0.1))
    return t.to(device), (torch.from_numpy(np.stack(ys)).float() / 255).unsqueeze(1).to(device)

def loss_fn(lg, y):
    bce = F.binary_cross_entropy_with_logits(lg, y)
    p = torch.sigmoid(lg); inter = (p*y).sum((1,2,3))
    dice = 1 - (2*inter + 1) / (p.sum((1,2,3)) + y.sum((1,2,3)) + 1)
    return bce + dice.mean()

opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
t0 = time.time(); it = 0; run = []
model.train()
while True:
    frac = (time.time() - t0) / BUDGET
    if frac >= 1: break
    for g in opt.param_groups: g["lr"] = BASE_LR * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
    x, y = get_batch()
    with torch.autocast(device_type=device, enabled=use_amp):
        lg = model(x)
    loss = loss_fn(lg.float(), y)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    run.append(loss.item()); it += 1
    if it % 50 == 0:
        print(f"iter {it} | loss {np.mean(run[-50:]):.4f} | {(time.time()-t0)/60:.1f}/{BUDGET/60:.0f} min")
torch.save(model.state_dict(), os.path.join(WORK_DIR, "model_v3.pt")); print("trained", it, "iterations")

resumed from ./work\model_v2.pt


C:\Users\shrut\AppData\Local\Temp\ipykernel_30496\3194013832.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


iter 50 | loss 0.4303 | 1.2/25 min
iter 100 | loss 0.3994 | 2.3/25 min
iter 150 | loss 0.4246 | 3.4/25 min
iter 200 | loss 0.4047 | 4.6/25 min
iter 250 | loss 0.4133 | 5.7/25 min
iter 300 | loss 0.3860 | 6.8/25 min
iter 350 | loss 0.3943 | 8.0/25 min


KeyboardInterrupt: 

In [ ]:
# 6) full-image inference (1024px) + validation probabilities
@torch.no_grad()
def predict_probs(X, D, bs=2):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = to_tensor(X[i:i+bs], D[i:i+bs]).to(device)
        with torch.autocast(device_type=device, enabled=use_amp):
            p = torch.sigmoid(model(x).float())
            if TTA: p = (p + torch.sigmoid(model(x.flip(-1)).float()).flip(-1)) / 2
        out.append(p.squeeze(1).cpu().numpy())
    return np.concatenate(out)

Pva_full = predict_probs(Xva, Dva)
Pva = np.stack([cv2.resize(p, (EVAL_RES, EVAL_RES), interpolation=cv2.INTER_AREA) for p in Pva_full])

In [ ]:
# 7) post-processing: components with confidence score, cached IoUs, wide fast grid search
def get_components(prob, thr, k, s=1.0):
    m = (prob > thr).astype(np.uint8)
    kk = int(round(k * s))
    if kk > 1: m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kk, kk)))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)   # fill holes
    m = np.zeros_like(m); cv2.drawContours(m, cnts, -1, 1, thickness=-1)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    return [(lab == i, int(stats[i, cv2.CC_STAT_AREA]), float(prob[lab == i].mean())) for i in range(1, n)]

def select(comps, min_area, min_score, s=1.0):
    return [c for c in comps if c[1] >= min_area * s * s and c[2] >= min_score]

def iou_mats(comps, annots):   # one (n_gt x n_pred) IoU matrix per annotator, computed once per (thr, k)
    nP = len(comps); out = []
    P = np.stack([c[0].ravel() for c in comps]).astype(np.float32) if nP else None
    for gts in annots:
        if nP and gts:
            G = np.stack([g.ravel() for g in gts]).astype(np.float32); inter = G @ P.T
            out.append(inter / (G.sum(1)[:, None] + P.sum(1)[None] - inter + 1e-9))
        else: out.append(np.zeros((len(gts), nP), np.float32))
    return out

# GT statistics, to sanity-check how many filaments to expect
n_gt = np.mean([len(g) for a in GTva for g in a]); print("GT filaments per image (per annotator): %.1f" % n_gt)

THRS, KS = [0.3, 0.4, 0.5, 0.6, 0.7], [0, 5, 9, 15, 25, 35]
MAS, MSS = [20, 40, 80, 150, 300, 600], [0.0, 0.5, 0.6, 0.7, 0.8]
results = []
for thr in tqdm(THRS):
    for k in KS:
        cache = []
        for prob, annots in zip(Pva, GTva):
            comps = get_components(prob, thr, k)
            cache.append((np.array([c[1] for c in comps]), np.array([c[2] for c in comps]), iou_mats(comps, annots)))
        for ma in MAS:
            for ms in MSS:
                s_iou = tp = fp = fn = 0
                for areas, scores, mats in cache:
                    keepm = (areas >= ma) & (scores >= ms) if len(areas) else np.zeros(0, bool)
                    npred = int(keepm.sum())
                    for iou in mats:
                        sub = iou[:, keepm]; mt = sub > 0.5; t = int(mt.sum())
                        s_iou += float(sub[mt].sum()); tp += t; fp += npred - t; fn += iou.shape[0] - t
                results.append((s_iou / max(tp + .5*fp + .5*fn, 1e-9), thr, k, ma, ms))
results.sort(reverse=True)
print("top 5 (PQ, thr, close_k, min_area, min_score):"); [print("  %.4f %s %s %s %s" % r) for r in results[:5]]
BEST_PQ, THR, K, MINA, MINS = results[0]
print("best val PQ: %.4f" % BEST_PQ)

GT filaments per image (per annotator): 7.2


100%|██████████| 5/5 [01:43<00:00, 20.61s/it]

top 5 (PQ, thr, close_k, min_area, min_score):
  0.3818 0.3 0 40 0.8
  0.3755 0.3 0 20 0.8
  0.3687 0.3 5 40 0.8
  0.3685 0.3 5 40 0.7
  0.3684 0.3 0 40 0.7
best val PQ: 0.3818


In [ ]:
# 8) predict test set -> submission_v3.csv
rows = []; n_img_with = 0
for stem, path in tqdm(test_files.items()):
    g = read_gray(path); H, W = g.shape
    x, d = make_inputs(g)
    prob = cv2.resize(predict_probs(x[None], d[None])[0], (W, H), interpolation=cv2.INTER_LINEAR)
    s = W / EVAL_RES
    insts = select(get_components(prob, THR, K, s), MINA, MINS, s)
    if not insts:   # fallback: biggest blob at a lower threshold
        insts = sorted(get_components(prob, 0.2, 0, s), key=lambda c: -c[1])[:1]
    n_img_with += bool(insts)
    for i, c in enumerate(insts, 1):
        r = mu.encode(np.asfortranarray(c[0].astype(np.uint8)))
        rows.append((f"{stem}_{i}", r["counts"].decode()))
sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv("submission_v3.csv", index=False, quoting=csv.QUOTE_NONE)
print(len(sub), "filaments (%.1f per image) over" % (len(sub)/len(test_files)), n_img_with, "/", len(test_files), "images -> submission_v3.csv")
sub.head()

100%|██████████| 180/180 [04:59<00:00,  1.66s/it]

1383 filaments (7.7 per image) over 180 / 180 images -> submission_v3.csv


,filament_id,segmentation_rle
0,20110120105534Ch_1,b`Ti23lo1200[O1WQN0hn12YQNKgn17XQNKen16[QNJen1...
1,20110120105534Ch_2,XjlP23ko13M2O1O1O1O1O0O1O2O000O100O010O010I7O1...
2,20110130110334Ch_1,]Xnm17bo19M2N2N1O101N3N1N3N1O2N1O2N1O3M1O2N1O1...
3,20110130110334Ch_2,d[lo13jo16J7I5K5G9nNoNoRNT1ol1nNoRNT1ol1nNnRNU...
4,20110130110334Ch_3,b_ld27ho14L5K3N3M2N100nPN[Ohn1Q1NO2O001N10O102...
